# HAT ×3 — Transformer 를 더 키우면

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

SwinIR 과 같은 계열이다. 창(window) 안에서만 보던 어텐션에 채널 어텐션과
창끼리 겹치는 어텐션을 더해 **보는 범위를 넓혔다**. 손실은 여전히 L1 하나다.

| | 구조 | 손실 | 파라미터 |
|---|---|---|---|
| EDSR | CNN (residual) | L1 | 1.55M |
| SRGAN | CNN + GAN | MSE + VGG + 적대적 | 0.77M |
| ESRGAN | CNN (RRDB) + GAN | L1 + VGG + RaGAN | 5.91M |
| SwinIR | Transformer (Swin) | L1 | 11.94M |
| **HAT** | **Transformer (Swin + 채널·중첩 어텐션)** | **L1** | **20.81M** |

## 1. 데이터

In [ ]:
import sys, urllib.request
!pip install -q timm einops

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'hat_arch.py', 'hat_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

val_lr, val_hr = pair('validation', REP['validation'])
test_lr = load_test()
show([('validation (Paris)', val_lr, val_hr), ('test (Incheon)', test_lr, None)])

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

SwinIR 보다 무거워서 128px 을 그대로 넣으면 12 GB 를 쓴다. Colab T4 로는 빠듯하다.
64px 씩 잘라 batch 1 로 돌린다 (약 3 GB). 실제 학습도 48px 씩 잘라서 했다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from hat_models import build_hat

# 자를 크기는 창 크기 16 의 배수여야 한다. HAT 이 입력을 16×16 창으로 쪼개 보기 때문이다.
N_TRAIN, EPOCHS, BATCH, CROP = 16, 1, 1, 64
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
x, y = to_t(lo)[:, :, :CROP, :CROP], to_t(hi)[:, :, :CROP * 3, :CROP * 3]
loader = DataLoader(TensorDataset(x, y), batch_size=BATCH, shuffle=True)

net = build_hat(3).to(dev).train()
opt = torch.optim.Adam(net.parameters(), 2e-4, betas=(0.9, 0.99))
crit = nn.L1Loss()

for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, yb in loader:
        loss = crit(net(xb.to(dev)), yb.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   L1 {tot/len(loader):.5f}')

del net, opt, loader, x, y
torch.cuda.empty_cache()      # 다음 셀에서 학습된 가중치를 올릴 자리를 비운다

## 3. 학습 로그

전체 데이터로 100 epoch 돌린 기록이다. SwinIR 과 같이 판별자가 없어 단조롭게 내려간다.

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/05_hat_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.l1, color='#2f6f9f', lw=1.6)
ax[0].set_title('Training L1 loss'); ax[0].set_ylabel('L1')

ax[1].plot(e.index, e.PSNR, color='#4f9d69', lw=1.6)
ax[1].set_title('Validation PSNR during training'); ax[1].set_ylabel('dB')

ax[2].step(e.index, e.lr, color='#c96a5b', lw=1.6, where='post')
ax[2].set_yscale('log'); ax[2].set_title('Learning rate (halved 3 times)')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'L1  {e.l1.iloc[0]:.5f} -> {e.l1.iloc[-1]:.5f}   ({(1-e.l1.iloc[-1]/e.l1.iloc[0])*100:.0f}% 감소)')

## 4. 결과

In [ ]:
from hat_models import load_hat, hat_upscale

net = load_hat(fetch(f'{MODEL}/checkpoints/hat_x3.pth', 'hat_x3.pth'))
print(f'HAT  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

upscale = lambda lr: hat_upscale(net, lr)

zoom([('Original LR', nearest(val_lr)), ('Bicubic', bicubic(val_lr)),
      ('HAT', upscale(val_lr)), ('Target HR', val_hr)],
     title='validation (Paris), x3')

## 5. 평가

In [ ]:
rows = compare(upscale, label='HAT')

## 6. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
t_bic = bicubic(test_lr)
t_sr = upscale(test_lr)

zoom([('Original LR', nearest(test_lr)), ('Bicubic', t_bic), ('HAT', t_sr)],
     ref=t_bic, title='test (Incheon), x3 - no target')

def sharpness(a):
    return float(cv2.Laplacian(cv2.cvtColor(a, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'{"":10s}{"sharpness":>11s}{"mean RGB":>22s}')
print(f'{"Bicubic":10s}{sharpness(t_bic):11.2f}{str(t_bic.reshape(-1,3).mean(0).round(1)):>22s}')
print(f'{"HAT":10s}{sharpness(t_sr):11.2f}{str(t_sr.reshape(-1,3).mean(0).round(1)):>22s}')

imageio.imwrite('incheon_hat.png', t_sr)
print('\nincheon_hat.png 저장')

## 7. 다섯 모델 정리

같은 검증 10패치, 같은 인천 사진으로 잰 값이다.

| 모델 | PSNR | SSIM | 인천 선명도 | 파라미터 |
|---|---|---|---|---|
| Bicubic | 18.15 | 0.4805 | 9.83 | — |
| EDSR | 18.97 | 0.5462 | 14.60 | 1.55M |
| SRGAN | 18.30 | 0.5187 | 22.03 | 0.77M |
| ESRGAN | 16.55 | 0.4208 | **41.59** | 5.91M |
| SwinIR | 19.04 | 0.5483 | 13.88 | 11.94M |
| **HAT** | **19.06** | **0.5505** | 14.07 | 20.81M |

**모델을 키운 만큼 오르지 않는다.** EDSR 에서 HAT 으로 파라미터를 13배 키웠는데
PSNR 은 0.09 dB 올랐다. SwinIR 에서 HAT 은 1.7배에 0.02 dB 다.

학습은 합성 저해상도로 하고 검증은 실제 Sentinel-2 로 한다. 이 차이가 병목이라
모델 용량으로는 넘지 못한다. 여섯 모델 전부에서 같은 벽이 보인다.

**지표와 선명도는 여전히 반대로 줄 선다.** PSNR 1·2위인 HAT·SwinIR 이 선명도는
최하위이고, PSNR 최하위인 ESRGAN 이 3배 선명하다.